# SV-PINN: 1D Poisson Equation — Example 1

This notebook reproduces Example 1 of:

> Marcondes, D. *Random test functions, H⁻¹ norm equivalence, and stochastic variational physics-informed neural networks.* ArXiv 2605.03542 (2025). https://arxiv.org/abs/2605.03542

It trains stochastic variational PINNs (SV-PINNs) on the 1D Poisson equation
$$-u''(x) = f(x), \quad x \in [0,1],$$
with exact solution
$$u(x) = \sin(2\pi x) + 0.1\sin(a\pi x) - x\bigl(\sin(2\pi) + 0.1\sin(a\pi)\bigr),$$
across frequency parameters $a \in \{1, 25, 50, 100, 150, 200, 250, 300\}$, three network types (`daff`, `ff`, plain MLP), and two optimizers (`LBFGS`, `GD`). For each combination, both a strong-form PINN (GD only) and a weak-form SV-PINN are trained.

**To run a different repetition:** change `rep` in the parameter cell and re-run from there. Results are saved to disk and skipped if already present.

**Disclaimer:** Generated with Claude AI based on the file in https://github.com/dmarcondes/JINNAX/tree/master/SV_PINN_examples

## 1. Installation

Install `jinnax` and all required dependencies directly from GitHub.

In [10]:
!pip install git+https://github.com/dmarcondes/jinnax.git
!pip install git+https://github.com/dmarcondes/genree.git
!pip install orthax jaxopt

  Cloning https://github.com/dmarcondes/jinnax.git to /tmp/pip-req-build-gwx27nfv
  Running command git clone --filter=blob:none --quiet https://github.com/dmarcondes/jinnax.git /tmp/pip-req-build-gwx27nfv
  Resolved https://github.com/dmarcondes/jinnax.git to commit b9711671e9d67c871f69a2f70834e2d8cc833e7f
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/dmarcondes/genree.git to /tmp/pip-req-build-oxjcpreg
  Running command git clone --filter=blob:none --quiet https://github.com/dmarcondes/genree.git /tmp/pip-req-build-oxjcpreg
  Resolved https://github.com/dmarcondes/genree.git to commit b9a03678bdee16269deaccc17296ba91d5e87d80
  Preparing metadata (setup.py) ... done


## 2. Imports and JAX configuration

Enable 64-bit precision throughout (required for numerical stability) and set matrix multiplication to highest precision.

In [11]:
import jax
import os
import jax.numpy as jnp
from jinnax import nn as nn
from jinnax import data as jd

jax.config.update("jax_default_matmul_precision", "highest")
jax.config.update("jax_enable_x64", True)

## 3. Simulation parameters

- `L`: domain upper bound (the domain is $[0, L]$)
- `N`: number of grid points for boundary/collocation data and for the Matérn test-function grid
- `d`: spatial dimension
- `bsize`: number of random test functions sampled for the weak-form loss
- `rep`: repetition index (change to 1 or 2 to run other repetitions)

In [12]:
L     = [1]
N     = 1024
d     = 1
bsize = 25000
rep   = 0    # change to 1 or 2 to run other repetitions

## 4. Training

For each combination of frequency parameter `a`, network type, and optimizer:

1. **Skip** if the result file already exists on disk.
2. **Define** the exact solution `u` and the forcing term `upp = -u''`.
3. **Define** the PDE residual operator `oper(u, x) = Δu(x) − f(x)` via automatic differentiation.
4. **Generate** training data (boundary + collocation points) and a test grid using `jd.generate_PINNdata`.
5. **Configure** the network architecture and hyperparameters according to the network type:
   - `daff`: domain-aware Fourier features; boundary term dropped (BCs encoded in the features); `q=0`
   - `ff`: random Fourier features; boundary term kept; `q=4`
   - `None`: plain MLP; boundary term kept; `q=4`
   - `LBFGS` always overrides `q=0` (no self-adaptive weights)
6. **Train** the strong-form PINN (GD only, `tau=0`) and the weak-form SV-PINN (`tau=0.1`), saving checkpoints every 5000 epochs.

In [ ]:
for a in [1, 25, 50, 100, 150, 200, 250, 300]:
    for type in ['daff', 'ff', 'None']:
        for opt in ['LBFGS', 'GD']:
            fn = 'Final_Example1_a' + str(a) + '_type' + str(type) + '_opt' + opt + '_rep' + str(rep)
            if not os.path.isfile(fn + '_weak_epoch' + str(5000).rjust(6, '0') + '.pickle'):
                print(fn)
                # Solution
                def u(x):
                    return jnp.sin(2 * jnp.pi * x) + 0.1 * jnp.sin(a * jnp.pi * x) - x * (jnp.sin(2 * jnp.pi) + 0.1 * jnp.sin(a * jnp.pi))
                # Forcing term (negative of the Laplacian of u)
                def upp(x):
                    return -4 * jnp.pi**2 * jnp.sin(2 * jnp.pi * x) - 0.1 * (a * jnp.pi)**2 * jnp.sin(a * jnp.pi * x)
                # PDE residual operator: Laplacian(u_theta) - f
                def oper(u, x):
                    H = jax.hessian(u)
                    return jax.vmap(lambda xi: jnp.trace(H(xi)))(x).reshape((x.shape[0], 1)) - upp(x)
                # Generate training data (boundary and collocation points) and test grid
                data = jd.generate_PINNdata(u=u, xl=0, xu=L, Nb=N, Nc=N)
                if type == 'daff':
                    data['boundary'] = None  # BCs are encoded in the DAFF eigenfunctions
                    fargs = [L]
                    q = 0
                    width = [1] + [64] + 3 * [512] + [1]
                    tau = 0.1
                elif type == 'ff':
                    fargs = [1, 10]
                    q = 4
                    width = [1] + [64] + 3 * [512] + [1]
                    tau = 0.1
                else:
                    type = None
                    fargs = None
                    q = 4
                    width = [1] + 3 * [512] + [1]
                    tau = 0.1
                if opt == 'LBFGS':
                    q = 0  # LBFGS does not use self-adaptive weights
                test_data = jd.generate_PINNdata(u=u, xl=0, xu=L, Ns=2 * N)
                # Strong-form PINN (GD only)
                if opt != 'LBFGS':
                    resS = nn.train_SV_PINN(data, width.copy(), oper, test_data, tau=0, epochs=5001, at_each=5000, epoch_print=100, save=True, file_name=fn + '_strong', lr=0.001, exp_decay=True, transition_steps=100, decay_rate=0.9, mlp=True, key=rep, ftype=type, fargs=fargs, q=q, opt=opt, float64=True)
                # Weak-form SV-PINN
                resW = nn.train_SV_PINN(data, width.copy(), oper, test_data, resample=False, d=d, N=N, L=L, alpha=1, kappa=1, tau=tau, bsize=bsize, epochs=5001, at_each=5000, epoch_print=100, save=True, file_name=fn + '_weak', lr=0.001, exp_decay=True, transition_steps=100, decay_rate=0.9, mlp=True, ftype=type, fargs=fargs, q=q, key=rep, opt=opt, float64=True)

Final_Example1_a1_typedaff_optLBFGS_rep0
--------- LBFGS OPTIMIZER ---------
on 0: Time: 21 s Loss: 0.016798 L2 error: 110.455529
on 100: Time: 199 s Loss: 1e-06 L2 error: 0.750213
